# 01 - Understanding the data

Goal: before cleaning anything, see what is actually in the export. Arjun says refunds are > 1 crore/quarter, helpdesk says ~11 lakh/quarter. Want to find out why.

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 80)

tickets = pd.read_csv('tickets.csv')
agents = pd.read_csv('agents.csv')
orders = pd.read_csv('orders.csv')
customers = pd.read_csv('customers.csv')
products = pd.read_csv('products.csv')

for name, df in [('tickets', tickets), ('agents', agents), ('orders', orders),
                 ('customers', customers), ('products', products)]:
    print(f'{name:10s} {df.shape}')

tickets    (12238, 21)
agents     (44, 8)
orders     (15000, 8)
customers  (9500, 6)
products   (14, 7)


In [2]:
tickets.head(3)

,ticket_id,created_at,first_response_at,resolved_at,status,channel,customer_id,order_id,product_sku,category,priority,assigned_team,agent_id,transfers,csat_score,refund_amount_inr,refund_reason_code,replacement_issued,customer_message,agent_notes,source_system
0,TK-240001,2025-01-01 09:17,2025-01-01 09:20,2025-01-02 09:37,resolved,chat,C103445,VR894176,VA-EB-AIR,Returns & Refunds,High,Returns Desk,A3038,0,5.0,NaN,NaN,N,sir ji\npickup scheduled but no one showed up\nkal se\naiirlite VR894176,1. Pickup missed\n2. Checked pickup AWB\n3. rfnd processed without pickup as...,legacy_fd
1,TK-240002,2025-01-01 09:45,2025-01-01 10:18,2025-01-01 10:30,resolved,voice,C108122,VR897663,VA-SW-FIT,Account & Login,High,Voice Frontline,A3023,0,3.0,NaN,NaN,N,Really frustrating. Order VR897663 (my Nexa Fit band). cannot login to my ac...,"reported: unable to log in. chk otp logs. number updated, rslvd. -RG",helpdesk
2,TK-240002,2025-01-01 09:45,2025-01-01 10:18,2025-01-01 10:30,resolved,voice,C108122,VR897663,VA-SW-FIT,Account & Login,High,Voice Frontline,A3023,0,3.0,NaN,NaN,N,Really frustrating. Order VR897663 (my Nexa Fit band). cannot login to my ac...,"reported: unable to log in. chk otp logs. number updated, rslvd. -RG",legacy_fd


In [3]:
# missing values per column
tickets.isna().sum().to_frame('nulls').assign(pct=lambda d: (d.nulls / len(tickets) * 100).round(1))

,nulls,pct
ticket_id,0,0.0
created_at,0,0.0
first_response_at,0,0.0
resolved_at,622,5.1
status,0,0.0
channel,0,0.0
customer_id,0,0.0
order_id,4127,33.7
product_sku,0,0.0
category,0,0.0


## Basic distributions

In [4]:
for col in ['status', 'channel', 'category', 'priority', 'assigned_team', 'source_system', 'replacement_issued']:
    print(tickets[col].value_counts(dropna=False), '\n')

status
resolved    10518
closed       1098
open          358
pending       264
Name: count, dtype: int64 

channel
chat      5150
email     3883
voice     1969
social    1236
Name: count, dtype: int64 

category
Delivery & Shipping    2160
Other                  1864
Billing & Payments     1706
Returns & Refunds      1230
Connectivity           1079
Charging & Battery      971
App & Firmware          833
Audio Quality           730
Product Enquiry         651
Warranty & Repair       620
Account & Login         394
Name: count, dtype: int64 

priority
Normal    8689
High      2225
Low       1324
Name: count, dtype: int64 

assigned_team
Chat Frontline            3404
Logistics                 2160
Email Frontline           2062
Billing                   1706
Returns Desk              1230
Voice Frontline           1056
Escalations & Warranty     620
Name: count, dtype: int64 

source_system
helpdesk     8364
legacy_fd    3874
Name: count, dtype: int64 

replacement_issued
N    10891
Y  

In [5]:
tickets['created_at'] = pd.to_datetime(tickets['created_at'])
print('date range:', tickets.created_at.min(), '->', tickets.created_at.max())

# policy s9: new helpdesk went live 14 Sep 2025, so legacy rows should all be before that
print(tickets.groupby('source_system').created_at.agg(['min', 'max']))

date range: 2025-01-01 09:17:00 -> 2026-06-30 23:14:00
                              min                 max
source_system                                        
helpdesk      2025-01-01 09:45:00 2026-06-30 23:14:00
legacy_fd     2025-01-01 09:17:00 2025-09-14 01:19:00


## Duplicates (Sameer: "some tickets appear twice from the migration re-import")

In [6]:
print('rows:', len(tickets), '| unique ticket_ids:', tickets.ticket_id.nunique())

dup_ids = tickets[tickets.ticket_id.duplicated(keep=False)]
print('duplicated ids:', dup_ids.ticket_id.nunique())

# is every duplicate a helpdesk + legacy_fd pair?
dup_ids.groupby('ticket_id').source_system.apply(lambda s: tuple(sorted(s))).value_counts()

rows: 12238 | unique ticket_ids: 11600
duplicated ids: 638


source_system
(helpdesk, legacy_fd)    638
Name: count, dtype: int64

In [7]:
# which columns actually differ between the two copies?
cols = [c for c in tickets.columns if c not in ('source_system',)]
diff_counts = {}
for tid, g in dup_ids.groupby('ticket_id'):
    a, b = g.iloc[0], g.iloc[1]
    for c in cols:
        if not (a[c] == b[c] or (pd.isna(a[c]) and pd.isna(b[c]))):
            diff_counts[c] = diff_counts.get(c, 0) + 1
diff_counts

{'refund_amount_inr': 125}

## Refund amounts - where does the crore come from?

In [8]:
refunds = tickets[tickets.refund_amount_inr.notna()]
print('refund rows:', len(refunds))
print('raw total (what Arjun sums): Rs', f'{refunds.refund_amount_inr.sum():,.0f}')

refunds.groupby('source_system').refund_amount_inr.describe()

refund rows: 2465
raw total (what Arjun sums): Rs 230,124,081


,count,mean,std,min,25%,50%,75%,max
source_system,,,,,,,,
helpdesk,1690.0,2850.817160,2135.919450,32.0,1299.0,2499.0,3499.0,13998.0
legacy_fd,775.0,290717.677419,232365.900167,5700.0,126200.0,224900.0,449900.0,1399800.0


In [9]:
# legacy median looks ~100x helpdesk. check on duplicate pairs that have a refund in both copies
pairs = (dup_ids[dup_ids.refund_amount_inr.notna()]
         .pivot_table(index='ticket_id', columns='source_system', values='refund_amount_inr')
         .dropna())
pairs['ratio'] = pairs['legacy_fd'] / pairs['helpdesk']
print(pairs.ratio.value_counts())
pairs.head()

ratio
100.0    125
Name: count, dtype: int64


source_system,helpdesk,legacy_fd,ratio
ticket_id,,,
TK-240003,900.0,90000.0,100.0
TK-240009,2124.0,212400.0,100.0
TK-240091,600.0,60000.0,100.0
TK-240121,1999.0,199900.0,100.0
TK-240134,1999.0,199900.0,100.0


Ratio is exactly 100 -> legacy Freshdesk stored paise (policy s9: "legacy tool stored monetary values in its own native unit").

In [10]:
# quick what-if: fix paise + drop duplicates (keep helpdesk copy), see if it lands near helpdesk's ~11L/quarter
t = tickets.copy()
t['amount'] = np.where(t.source_system == 'legacy_fd', t.refund_amount_inr / 100, t.refund_amount_inr)
t = t.sort_values('source_system').drop_duplicates('ticket_id', keep='first')  # 'helpdesk' sorts before 'legacy_fd'

print('clean total: Rs', f'{t.amount.sum():,.0f}')
t.groupby(t.created_at.dt.to_period('Q')).amount.sum().map('{:,.0f}'.format)

clean total: Rs 6,709,932


created_at
2025Q1      609,583
2025Q2      727,422
2025Q3    1,207,091
2025Q4    1,627,575
2026Q1    1,258,438
2026Q2    1,279,823
Freq: Q-DEC, Name: amount, dtype: object

## Reason codes

In [11]:
r = t[t.amount.notna()]
r.groupby('refund_reason_code').amount.agg(['count', 'sum']).assign(
    pct_count=lambda d: (d['count'] / d['count'].sum() * 100).round(1)
).sort_values('count', ascending=False)

,count,sum,pct_count
refund_reason_code,,,
GW-OTHER,991,2907036.0,42.4
RETURN-QC-OK,452,1181386.0,19.3
DUP-PAYMENT,321,884586.0,13.7
CANCEL,222,612950.0,9.5
DOA-REPL,147,471190.0,6.3
WTY-BUYBACK,89,289154.0,3.8
PRICE-ADJ,71,207073.0,3.0
LOST-TRANSIT,47,156557.0,2.0


In [12]:
# goodwill cap is Rs 500 (policy s5) - how many GW-OTHER refunds are above it?
gw = r[r.refund_reason_code == 'GW-OTHER']
print('GW-OTHER refunds:', len(gw), '| above 500:', (gw.amount > 500).sum())
gw[gw.amount > 500].groupby(gw.created_at.dt.to_period('Q')).amount.agg(['count', 'sum'])

GW-OTHER refunds: 991 | above 500: 879


,count,sum
created_at,,
2025Q1,63,229530.0
2025Q2,87,279465.0
2025Q3,154,516795.0
2025Q4,221,752692.0
2026Q1,166,514664.0
2026Q2,188,578486.0


## Refund AND replacement on same ticket (policy says never)

In [13]:
both = r[r.replacement_issued == 'Y']
print('refund + replacement:', len(both), 'of', len(r), 'refunds')
print(both.groupby(both.created_at.dt.to_period('Q')).size())
both.assigned_team.value_counts()

refund + replacement: 166 of 2340 refunds
created_at
2025Q1    15
2025Q2    14
2025Q3    27
2025Q4    35
2026Q1    43
2026Q2    32
Freq: Q-DEC, dtype: int64


assigned_team
Chat Frontline            50
Logistics                 48
Email Frontline           22
Voice Frontline           19
Returns Desk              13
Escalations & Warranty    11
Billing                    3
Name: count, dtype: int64

## Agents

In [14]:
print('roster rows:', len(agents), '| unique agent_ids:', agents.agent_id.nunique())
print(agents.team.value_counts())

# any agent ids in tickets that aren't on the roster?
missing = set(t.agent_id.dropna()) - set(agents.agent_id)
print('agent ids not in roster:', missing)

roster rows: 44 | unique agent_ids: 44
team
Chat Frontline            15
Email Frontline            7
Escalations & Warranty     6
Logistics                  5
Voice Frontline            4
Billing                    4
Returns Desk               3
Name: count, dtype: int64
agent ids not in roster: set()


In [15]:
# policy s6: returns desk should do the 'large majority' of refunds. does it?
r_team = r.merge(agents[['agent_id', 'team']], on='agent_id', how='left')
r_team.groupby('team').amount.agg(['count', 'sum']).assign(
    share=lambda d: (d['count'] / d['count'].sum() * 100).round(1)
).sort_values('count', ascending=False)

,count,sum,share
team,,,
Returns Desk,612,1692666.0,26.2
Billing,592,1579977.0,25.3
Chat Frontline,387,1165339.0,16.5
Logistics,356,1111749.0,15.2
Email Frontline,213,564316.0,9.1
Voice Frontline,98,297150.0,4.2
Escalations & Warranty,82,298735.0,3.5


## Joins to reference data

In [16]:
print('order_id missing:', t.order_id.isna().mean().round(3))

# customers often type the order id in their message, e.g. 'vr896352'
found = t.loc[t.order_id.isna(), 'customer_message'].str.extract(r'(VR\d{6})', flags=2)[0]  # flags=2 -> ignore case
print('recoverable from message:', found.notna().sum(), 'of', t.order_id.isna().sum())

print('skus not in products:', set(t.product_sku.dropna()) - set(products.sku))
print('customers not in customers.csv:', len(set(t.customer_id) - set(customers.customer_id)))

order_id missing: 0.336
recoverable from message: 212 of 3899
skus not in products: set()
customers not in customers.csv: 0


## Free text - what the AI will have to read

In [17]:
gw[['ticket_id', 'amount', 'customer_message', 'agent_notes']].sample(6, random_state=1)

,ticket_id,amount,customer_message,agent_notes
5528,TK-246152,1799.0,Very poor quality. Ordr VR899685 (airlite earbuds). still waiting for my ref...,1. Refund delay\n2. chk refund status on PG\n3. Refunded Rs 1799
5153,TK-245664,1424.0,hi\nu picked up the item 19 days ago & my money hasn't come back,cx reached out - rfnd delay. chk reverse pickup QC status. rfnd of Rs 1424 i...
11938,TK-254155,2799.0,[IVR transcript] this is regarding pulse 2. cancel order. i wannt a replacem...,as discussed ~Amit
5886,TK-246601,1187.0,hey\npayment went through but i got no order id\nthe mini speaker VR907826\n...,1. amount deducted without ord\n2. chk pg dashboard for txn\n3. full rfnd pr...
8591,TK-249993,1799.0,hi\nreturn pickup has not happened\nthe fit band\nurgent,"full refund processed (1799)\nCx asked for replacement + refund, explained p..."
10840,TK-252794,1899.0,"Dear Sir/Madam,\n\nI am writing with reference to my order of my Nexa Fit ba...",Cx says rfnd pending. chk reverse picukp QC status. Full rfnd proessed (1899...


In [18]:
# notes that talk about a refund but refund_amount is blank
no_amt = t[t.amount.isna() & t.agent_notes.str.contains(r'refund|rfnd|refnd', case=False, na=False)]
print(len(no_amt))
no_amt[['ticket_id', 'category', 'agent_notes']].sample(6, random_state=1)

1192


,ticket_id,category,agent_notes
7662,TK-248826,Other,issue: shipment not received | raised ticket with courier partner -> refund ...
5929,TK-246656,Returns & Refunds,"cx reached out - pickup not done. Checked pickup AWB. Pickup done, refund in..."
9243,TK-250818,Billing & Payments,cx recahed out - failed ord after payment. Asked for UTR. Auto-rfnd conf wit...
3782,TK-244076,Billing & Payments,cx reached out - amount deducted without ord. conf w/ payment gateway. rfnd ...
3422,TK-243694,Delivery & Shipping,"cancel odrer | chk shipment status -> cancelled, rfnd initiated"
10560,TK-252457,Other,delivery delayed | checked awb with courier -> refund initiated as lost in t...


## CSAT check (Priya: "CSAT went up 0.4" after Q4 change)

In [19]:
# blanks = no response, excluded (policy s8). mean() skips NaN already
t.groupby(t.created_at.dt.to_period('Q')).csat_score.agg(['mean', 'count']).round(2)

,mean,count
created_at,,
2025Q1,3.54,476
2025Q2,3.52,603
2025Q3,3.48,826
2025Q4,3.51,1242
2026Q1,3.49,1051
2026Q2,3.46,1039


## Notes so far

(fill in after running)